<a href="https://colab.research.google.com/github/darkamgel/CSV-Reproducibilty/blob/main/CSC786_Ethical_Reproducible_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 786 — Data Ethics and Reproducibility Workshop

**Project:** Ethical and Reproducible Weather Data Collection  
**Environment:** Google Colab, Python, GitHub  
**Data Source:** WeatherAPI

## 1. Introduction

In this notebook, I demonstrate how to collect weather data from an external API while following ethical data management, security, transparency, and reproducibility practices.

I use Google Colab for executing Python code, GitHub for version control, and WeatherAPI for retrieving weather observations and forecasts.

My objective is not simply to collect weather data, but also to document the complete process so that another student can independently reproduce my work.

Throughout the notebook, I explain the purpose of each step, the tools I use, the data I collect, and how I verify the results.

## 2. Project Objectives

My objectives are to:

1. Set up a GitHub repository and connect it to Google Colab.
2. Document the Python environment and required dependencies.
3. Securely configure authentication credentials using Google Colab Secrets.
4. Retrieve weather data from WeatherAPI.
5. Process the API response and export structured CSV datasets.
6. Record collection timestamps, request parameters, and data provenance.
7. Generate SHA-256 checksums to verify dataset integrity.
8. Document ethical considerations, privacy, security, and limitations.
9. Commit and push research artifacts to GitHub.
10. Provide detailed instructions for independent peer reproduction.

## 3. Research Workflow

I organize this notebook into the following stages:

**Stage 1:** GitHub repository setup and connection  
**Stage 2:** Environment configuration  
**Stage 3:** Documentation and ethical considerations  
**Stage 4:** API configuration and data collection  
**Stage 5:** Data processing and CSV export  
**Stage 6:** Provenance metadata and integrity verification  
**Stage 7:** GitHub commits and version control  
**Stage 8:** Peer reproduction and final verification

## 4. Instructions for Peer Reviewers

I designed this notebook so that another student can reproduce the experiment without requiring access to my personal credentials.

If you are reviewing my work, follow these steps:

1. Open my GitHub repository and read the README.md file.
2. Open the notebook in Google Colab.
3. Start a fresh Colab runtime.
4. Obtain your own WeatherAPI key from https://www.weatherapi.com/.
5. Add the key to Google Colab Secrets using the name `WEATHERAPI_KEY` and enable notebook access.
6. Follow the notebook instructions and execute the reproducibility cells in their documented order.
7. Verify that the API request succeeds and the expected data columns are generated.
8. Check that dataset validation, metadata generation, and SHA-256 verification succeed.
9. Compare your generated file structure and results with the expectations documented in my repository.

### GitHub Authentication for Peer Review

The GitHub connection and push steps are part of my development workflow.

You do not need my GitHub token to reproduce the experiment. You can use the public repository to obtain the notebook and run the data collection and validation steps independently.

If you want to reproduce the GitHub commit and push workflow, create your own repository and configure your own `GITHUB_TOKEN` in Colab Secrets.

Do not attempt to push changes to my repository.

### Expected Differences

Because WeatherAPI returns time-dependent weather data, your measurements may differ from mine.

This does not necessarily indicate an error. The important checks are whether the notebook executes correctly, the expected dataset structure is produced, the documentation is complete, and the integrity verification succeeds.

## 5. Security and Ethical Transparency

I do not intentionally expose API keys or GitHub tokens in my source code, CSV datasets, metadata, or repository documentation.

I use Google Colab Secrets to manage sensitive credentials.

I document the source of the data, collection procedure, processing steps, validation checks, and known limitations.

Before publishing weather data, I also consider the data provider's applicable terms and redistribution restrictions.

## 6. References

- WeatherAPI Documentation: https://www.weatherapi.com/docs/
- Google Colab: https://colab.research.google.com/
- Git Documentation: https://git-scm.com/doc
- Python Documentation: https://docs.python.org/3/
- NIST Secure Hash Standard: https://doi.org/10.6028/NIST.FIPS.180-4

## Step 2 — GitHub Repository Setup and Authentication

### 2.1 Create a GitHub Repository

I first create a public GitHub repository to store my notebook, source code, documentation, and reproducibility artifacts.

1. Open [GitHub — New Repository](https://github.com/new).
2. Configure the repository using the following settings:
   - **Repository Name:** `CSC786-AI-Ethics-Lab`
   - **Visibility:** Public
   - **Initialize README:** No
   - **Add .gitignore:** No
   - **Choose a License:** No
3. Click **Create repository**.


### 2.2 Generate a GitHub Personal Access Token

To connect Google Colab with GitHub, I use a fine-grained Personal Access Token (PAT).

1. Open [GitHub Personal Access Token Settings](https://github.com/settings/personal-access-tokens/new).
2. Select my GitHub account as the resource owner.
3. Give the token a descriptive name, such as `CSC786-Colab-Access`.
4. Set an appropriate expiration date.
5. Under **Repository access**, select **Only select repositories**.
6. Choose the repository created in Step 2.1.
7. Under **Repository permissions**, set **Contents** to **Read and write**.
8. Generate the token and copy it securely.

### 2.3 Store the GitHub Token in Google Colab Secrets

Instead of hardcoding my GitHub token in the notebook, I store it securely using Google Colab Secrets.

1. Open my notebook in Google Colab.
2. Click the **Secrets** icon in the left sidebar.
3. Select **Add new secret**.
4. Enter `GITHUB_TOKEN` as the secret name.
5. Paste my GitHub personal access token into the value field.
6. Enable **Notebook access**.

### 2.4 Security and Peer Reproducibility

I do not include my GitHub token in notebook source code, CSV files, metadata, or GitHub commits.

For independent reproduction, peer reviewers can clone or download my public repository without a GitHub token.

If a reviewer wants to reproduce the GitHub publishing steps, they should create their own repository and configure their own `GITHUB_TOKEN` through Google Colab Secrets.

This approach protects my credentials while allowing another researcher to reproduce the experiment independently.

In [4]:
# GitHub Configuration

from google.colab import userdata
from pathlib import Path
import os
import base64
import subprocess

GITHUB_USERNAME = "darkamgel"
REPO_NAME = "CSV-Reproducibilty"

REPO_URL = (
    f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
)

ROOT = Path("/content") / REPO_NAME
DATA = ROOT / "data"

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

assert GITHUB_TOKEN, "Add GITHUB_TOKEN to Colab Secrets."

# Configure authentication without placing a token in the URL
auth = base64.b64encode(
    f"x-access-token:{GITHUB_TOKEN}".encode()
).decode()

os.environ["GIT_CONFIG_COUNT"] = "1"
os.environ["GIT_CONFIG_KEY_0"] = (
    "http.https://github.com/.extraheader"
)
os.environ["GIT_CONFIG_VALUE_0"] = (
    f"AUTHORIZATION: basic {auth}"
)

print("GitHub credentials loaded securely.")
print("Repository:", REPO_NAME)

GitHub credentials loaded securely.
Repository: CSV-Reproducibilty


In [5]:
# Clone GitHub Repository

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(ROOT)],
        check=True
    )
else:
    print("Repository directory already exists.")

assert (ROOT / ".git").exists(), (
    "Repository is not a valid Git repository."
)

os.chdir(ROOT)

subprocess.run(
    ["git", "config", "user.name", GITHUB_USERNAME],
    check=True
)

subprocess.run(
    [
        "git", "config", "user.email",
        "YOUR_GITHUB_EMAIL"
    ],
    check=True
)

print("GitHub repository connected successfully.")
print("Working directory:", ROOT)

GitHub repository connected successfully.
Working directory: /content/CSV-Reproducibilty


In [6]:
# GitHub Commit and Push

os.chdir(ROOT)

# Create initial README
readme = ROOT / "README.md"

if not readme.exists():
    readme.write_text(
        "# CSC 786 - Data Ethics and Reproducibility\n\n"
        "Ethical and reproducible weather data collection.\n",
        encoding="utf-8"
    )

# Exclude credentials and temporary files
gitignore = ROOT / ".gitignore"

gitignore.write_text(
    ".env\n"
    ".env.*\n"
    "*.key\n"
    "credentials.json\n"
    ".ipynb_checkpoints/\n"
    "__pycache__/\n",
    encoding="utf-8"
)

subprocess.run(
    ["git", "add", "README.md", ".gitignore"],
    check=True
)

subprocess.run(
    ["git", "commit", "-m", "Initial reproducibility setup"],
    check=True
)

subprocess.run(
    ["git", "branch", "-M", "main"],
    check=True
)

subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    check=True
)

print("Initial GitHub push successful.")

Initial GitHub push successful.


## 3. Python Environment Setup

### 3.1 Objective

In this section, I configure the Python environment required for my data collection experiment. I also document the libraries and software versions used so that another student can reproduce my work.

I use Google Colab as my execution environment because it provides a Python runtime without requiring local installation.

### 3.2 Required Libraries

The experiment uses the following Python libraries:

- **pandas:** To process and export structured weather data.
- **requests:** To communicate with the weather API through HTTP requests.
- **json:** To manage JSON data and metadata.
- **hashlib:** To generate SHA-256 checksums for dataset integrity.
- **pathlib:** To manage files and directories.
- **datetime:** To record collection timestamps.
- **platform:** To document the execution environment.

### 3.3 Reproducibility Instructions

If you are reviewing my notebook:

1. Open the notebook in Google Colab.
2. Select **Runtime → Change runtime type → Python 3**.
3. Use a fresh runtime and execute the environment setup cells in order.
4. Verify that the required libraries load successfully.
5. Compare the recorded package versions with the `requirements.txt` file.

The Python standard-library modules do not require separate installation.

In [8]:
# Create requirements.txt

from importlib.metadata import version

requirements = (
    f"pandas=={version('pandas')}\n"
    f"requests=={version('requests')}\n"
)

requirements_path = ROOT / "requirements.txt"

requirements_path.write_text(
    requirements,
    encoding="utf-8"
)

print("requirements.txt created successfully.")
print("-" * 50)
print(requirements)# Python Environment Setup

import os
import sys
import json
import hashlib
import platform
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import requests

# ROOT and DATA are established from GitHub setup.
# Do not overwrite ROOT with another directory.

DATA = ROOT / "data"

ROOT.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)

print("Environment initialized successfully.")
print("-" * 50)
print("Python version:", platform.python_version())
print("Operating system:", platform.system())
print("Pandas version:", pd.__version__)
print("Requests version:", requests.__version__)
print("Project directory:", ROOT)
print("Data directory:", DATA)

requirements.txt created successfully.
--------------------------------------------------
pandas==2.2.3
requests==2.32.4

Environment initialized successfully.
--------------------------------------------------
Python version: 3.13.16
Operating system: Linux
Pandas version: 2.2.3
Requests version: 2.32.4
Project directory: /content/CSV-Reproducibilty
Data directory: /content/CSV-Reproducibilty/data


In [9]:
# Verify Python Environment

assert ROOT.exists(), "Repository directory missing."
assert (ROOT / ".git").exists(), "Git repository missing."
assert DATA.exists(), "Data directory missing."
assert requirements_path.exists(), "requirements.txt missing."

required_packages = ["pandas", "requests"]

for package in required_packages:
    print(f"PASS: {package} == {version(package)}")

print("-" * 50)
print("Environment verification PASSED.")

PASS: pandas == 2.2.3
PASS: requests == 2.32.4
--------------------------------------------------
Environment verification PASSED.


In [10]:
# Commit and Push Environment Setup
%cd {ROOT}

!git add .
!git commit -m "Update environment setup"
!git push

/content
[main e52c2b0] Update environment setup
 1 file changed, 2 insertions(+)
 create mode 100644 requirements.txt
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 362 bytes | 362.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/darkamgel/CSV-Reproducibilty.git
   5279078..e52c2b0  main -> main


## 4. Research Documentation and Ethical Transparency

### 4.1 Objective

In this section, I create the documentation necessary to make my experiment transparent, understandable, and reproducible.

I document the research purpose, data source, methodology, ethical considerations, and expected outputs before collecting the data.

My documentation includes three files:

- **README.md:** Describes my project, software requirements, setup instructions, and procedures for independent reproduction.
- **ETHICS.md:** Documents my approach to privacy, credential security, responsible API usage, and research limitations.
- **DATA_README.md:** Describes the dataset structure, data collection procedure, and verification methods.

### 4.2 Reproducibility

I provide detailed instructions so another student can run the notebook in Google Colab using their own API credentials.

Peer reviewers do not need access to my GitHub account or personal credentials to reproduce the data collection and verification procedures.

I will update the dataset documentation after data collection to include the actual filename, collection timestamp, dataset dimensions, and SHA-256 checksum.

In [11]:
# Create Project README

readme = """# CSC 786 - Ethical and Reproducible Data Collection

## Project Overview

In this project, I demonstrate a reproducible workflow
for collecting weather data using Google Colab, Python,
WeatherAPI, and GitHub.

I focus on ethical transparency, secure authentication,
data provenance, documentation, and reproducibility.

## Tools

- Google Colab
- Python 3
- Git and GitHub
- WeatherAPI
- pandas
- requests

## Data Source

Provider: WeatherAPI
Documentation: https://www.weatherapi.com/docs/
Endpoint: https://api.weatherapi.com/v1/forecast.json
Location: Sioux Falls, South Dakota
Forecast duration: 1 day

## Repository Contents

- CSC786_Ethical_Reproducible_Data_Collection.ipynb
- README.md
- ETHICS.md
- DATA_README.md
- requirements.txt
- .gitignore

Generated CSVs and provenance metadata are stored locally
in the data/ directory.

## Instructions for Peer Reviewers

I have designed this project for independent reproduction.

### Step 1: Obtain the Notebook

Open my public GitHub repository and download the notebook.

You can open it in Google Colab using File > Upload notebook.

### Step 2: Prepare the Environment

1. Start a fresh Google Colab runtime.
2. Select Python 3.
3. Run the environment setup cells.
4. Verify that the required libraries are available.

### Step 3: Configure WeatherAPI

1. Visit https://www.weatherapi.com/
2. Create your own account.
3. Obtain your own API key.
4. Open Google Colab Secrets.
5. Create a secret named WEATHERAPI_KEY.
6. Paste your API key.
7. Enable notebook access.

Never place credentials directly inside notebook code.

### Step 4: Run the Experiment

Run the data collection and processing cells in order.

The notebook will:
1. Retrieve weather data.
2. Validate the API response.
3. Extract hourly forecasts.
4. Create a structured DataFrame.
5. Export a CSV file.
6. Generate provenance metadata.
7. Calculate and verify SHA-256 checksums.

### Step 5: Verify Reproducibility

Check that:
- The API request succeeds.
- The expected dataset columns are present.
- Dataset validation passes.
- The CSV is generated.
- The metadata JSON is generated.
- SHA-256 verification passes.

Live weather values may differ between executions.

## GitHub Authentication

The GitHub connection and publishing steps are part of
my development workflow.

Peer reviewers may skip publishing steps.

If a reviewer wants to reproduce the Git operations,
they should use their own repository and GITHUB_TOKEN.

## Ethical Considerations

I do not publish authentication credentials.

I document my data sources and collection methodology.

I follow the provider's applicable usage conditions.

## Limitations

Weather forecasts change over time.

API access may depend on account limits and availability.

Reproduction focuses on workflow correctness and
data integrity rather than identical weather values.

## References

WeatherAPI: https://www.weatherapi.com/docs/
Google Colab: https://colab.research.google.com/
Python: https://docs.python.org/3/
Git: https://git-scm.com/doc
"""

(ROOT / "README.md").write_text(readme, encoding="utf-8")

print("README.md created successfully.")

README.md created successfully.


In [12]:
# Create ETHICS.md

ethics = """# Ethical Considerations

## Research Purpose

I use weather data to demonstrate ethical and
reproducible API data collection.

My objective is to document the research process
rather than make operational weather predictions.

## Privacy

I collect city-level weather forecasts.

I do not intentionally collect personal information,
user identifiers, or private communications.

## Credential Security

I store API keys and GitHub tokens using
Google Colab Secrets.

I do not intentionally publish credentials in
notebooks, CSV datasets, metadata, or GitHub commits.

## Responsible API Usage

I use documented API endpoints and limited requests.

I respect applicable provider rate limits,
account restrictions, and terms of service.

## Transparency

I document the data source, location, request parameters,
processing methods, collection timestamps, and limitations.

## Reproducibility

Other students can reproduce my procedure using their own
WeatherAPI credentials.

Because the data is time-dependent, reproduced values
may differ from my original results.

## Data Accuracy

Forecasts represent predictions and should not be
treated as independently verified observations.

## Data Sharing

Before publishing collected weather data, I must check
whether redistribution is permitted under the applicable
WeatherAPI terms and subscription plan.

## References

WeatherAPI Documentation:
https://www.weatherapi.com/docs/

WeatherAPI Terms:
https://www.weatherapi.com/terms.aspx
"""

(ROOT / "ETHICS.md").write_text(ethics, encoding="utf-8")

print("ETHICS.md created successfully.")

ETHICS.md created successfully.


In [13]:
# Create DATA_README.md

data_readme = """# Dataset Documentation

## Data Source

I use WeatherAPI to retrieve hourly weather forecasts
for Sioux Falls, South Dakota.

API Documentation:
https://www.weatherapi.com/docs/

## Collection Method

I use a Python HTTP GET request to retrieve JSON data.

I process the response using pandas and export
hourly forecast records into a CSV file.

## Expected Dataset Columns

| Column | Description |
|---|---|
| location | City |
| region | State or region |
| country | Country |
| forecast_time | Local forecast timestamp |
| temperature_c | Temperature in Celsius |
| humidity_percent | Relative humidity percentage |
| wind_kph | Wind speed in kilometers per hour |
| precipitation_mm | Precipitation in millimeters |
| condition | Text description of weather conditions |

## Data Validation

I will check:
- Required columns
- Missing values
- Valid timestamps
- Duplicate timestamps
- Numeric fields
- Valid humidity range
- Non-negative wind and precipitation

## Provenance

After collection, I will record:
- API endpoint
- Non-sensitive request parameters
- UTC collection timestamp
- Dataset filename
- Dataset dimensions
- Software versions
- SHA-256 checksum

## Reproducibility

A peer reviewer can run the notebook using their
own WeatherAPI credentials.

Live weather forecasts may differ between executions.

## Data Availability

Generated data is stored in the local Colab
data/ directory.

Public redistribution depends on the applicable
WeatherAPI usage terms.

## References

WeatherAPI:
https://www.weatherapi.com/docs/

pandas:
https://pandas.pydata.org/docs/
"""

(ROOT / "DATA_README.md").write_text(
    data_readme,
    encoding="utf-8"
)

print("DATA_README.md created successfully.")

DATA_README.md created successfully.


In [14]:
# Verify Documentation Files

required_files = [
    "README.md",
    "ETHICS.md",
    "DATA_README.md",
    "requirements.txt",
    ".gitignore"
]

for filename in required_files:
    path = ROOT / filename

    assert path.exists(), f"Missing file: {filename}"
    assert path.stat().st_size > 0, f"Empty file: {filename}"

    print(f"PASS: {filename}")

print("\nDocumentation verification successful.")

PASS: README.md
PASS: ETHICS.md
PASS: DATA_README.md
PASS: requirements.txt
PASS: .gitignore

Documentation verification successful.


In [15]:
%cd {ROOT}

!git add README.md ETHICS.md DATA_README.md
!git commit -m "Add research and ethical documentation"
!git push

/content
[main 70e5daa] Add research and ethical documentation
 3 files changed, 264 insertions(+), 2 deletions(-)
 create mode 100644 DATA_README.md
 create mode 100644 ETHICS.md
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.29 KiB | 3.29 MiB/s, done.
Total 5 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/darkamgel/CSV-Reproducibilty.git
   e52c2b0..70e5daa  main -> main


## 5. WeatherAPI Configuration and Data Collection

### 5.1 Objective

In this section, I configure WeatherAPI authentication and retrieve weather data for Sioux Falls, South Dakota.

I use WeatherAPI's forecast endpoint to collect weather information. The response contains current conditions and hourly forecasts, which I will process into a structured dataset.

### 5.2 API Configuration

- **Provider:** WeatherAPI
- **Documentation:** https://www.weatherapi.com/docs/
- **Endpoint:** `https://api.weatherapi.com/v1/forecast.json`
- **HTTP Method:** GET
- **Location:** Sioux Falls, South Dakota, USA
- **Forecast Duration:** 1 day
- **Response Format:** JSON

### 5.3 Secure API Key Setup

I use Google Colab Secrets to store my API key rather than including it directly in my code.

To configure the key:

1. Visit https://www.weatherapi.com/ and create an account.
2. Obtain an API key from the WeatherAPI dashboard.
3. Open Google Colab and select the **Secrets** icon in the left sidebar.
4. Create a secret named `WEATHERAPI_KEY`.
5. Paste the API key into the secret value field.
6. Enable notebook access for the secret.

### 5.4 Instructions for Peer Reviewers

If you are reproducing my experiment, you must use your own WeatherAPI key.

You do not need my credentials, and you should not modify the notebook to include your key directly.

After configuring the secret, execute the following cells in order.

The notebook will verify that authentication works, retrieve the weather data, and validate the API response.

### 5.5 Ethical Considerations

I use a documented API endpoint and make a limited request for one day of weather data.

I do not store API credentials in the collected dataset or GitHub repository.

Because the data is retrieved from a live API, another researcher may receive different measurements when running the same experiment.

**Reference:** https://www.weatherapi.com/docs/

In [17]:
# Load WeatherAPI Key from Google Colab Secrets

from google.colab import userdata

WEATHERAPI_KEY = userdata.get("WEATHERAPI_KEY")

assert WEATHERAPI_KEY, (
    "WEATHERAPI_KEY is missing. "
    "Add it to Google Colab Secrets and enable access."
)

print("WeatherAPI credential loaded successfully.")
print("Authentication method: Google Colab Secrets")
print("Credential value: [REDACTED]")

WeatherAPI credential loaded successfully.
Authentication method: Google Colab Secrets
Credential value: [REDACTED]


In [18]:
# Retrieve Weather Data from WeatherAPI

import requests

ENDPOINT = "https://api.weatherapi.com/v1/forecast.json"

LOCATION = "Sioux Falls, South Dakota"
FORECAST_DAYS = 1

PARAMS = {
    "key": WEATHERAPI_KEY,
    "q": LOCATION,
    "days": FORECAST_DAYS,
    "aqi": "no",
    "alerts": "no"
}

# Public parameters for future metadata
SAFE_PARAMS = {
    "q": LOCATION,
    "days": FORECAST_DAYS,
    "aqi": "no",
    "alerts": "no"
}

try:
    response = requests.get(
        ENDPOINT,
        params=PARAMS,
        timeout=30
    )

except requests.exceptions.Timeout:
    raise RuntimeError(
        "WeatherAPI request timed out."
    ) from None

except requests.exceptions.RequestException:
    raise RuntimeError(
        "Unable to connect to WeatherAPI."
    ) from None

# Avoid printing response URLs containing API keys
if response.status_code != 200:
    raise RuntimeError(
        f"WeatherAPI returned HTTP {response.status_code}. "
        "Check your API key and request limits."
    )

weather_data = response.json()

# Validate expected response structure
assert "location" in weather_data
assert "current" in weather_data
assert "forecast" in weather_data

assert weather_data["forecast"].get("forecastday"), (
    "Forecast data is missing."
)

print("WeatherAPI request successful.")
print("HTTP status:", response.status_code)
print("Location:", weather_data["location"]["name"])
print("Country:", weather_data["location"]["country"])
print("Forecast days:", len(
    weather_data["forecast"]["forecastday"]
))

WeatherAPI request successful.
HTTP status: 200
Location: Sioux Falls
Country: United States of America
Forecast days: 1


In [19]:
# Inspect WeatherAPI Response

location_info = weather_data["location"]
current = weather_data["current"]

print("WEATHER DATA SUMMARY")
print("-" * 50)

print("Location:", location_info["name"])
print("Region:", location_info["region"])
print("Country:", location_info["country"])
print("Last updated:", current["last_updated"])
print("Temperature (C):", current["temp_c"])
print("Humidity (%):", current["humidity"])
print("Wind speed (kph):", current["wind_kph"])
print("Condition:", current["condition"]["text"])

print(
    "Forecast date:",
    weather_data["forecast"]["forecastday"][0]["date"]
)

WEATHER DATA SUMMARY
--------------------------------------------------
Location: Sioux Falls
Region: South Dakota
Country: United States of America
Last updated: 2026-10-24 16:00
Temperature (C): 14.5
Humidity (%): 56
Wind speed (kph): 19.8
Condition: Patchy rain nearby
Forecast date: 2026-10-24


In [20]:
# Verify WeatherAPI Data

required_current_fields = [
    "last_updated",
    "temp_c",
    "humidity",
    "wind_kph",
    "condition"
]

for field in required_current_fields:
    assert field in current, (
        f"Missing current weather field: {field}"
    )

forecast_days = weather_data["forecast"]["forecastday"]

assert len(forecast_days) >= 1, (
    "No forecast days returned."
)

for day in forecast_days:
    assert "hour" in day, "Hourly forecast missing."
    assert len(day["hour"]) > 0, (
        "No hourly records returned."
    )

print("API RESPONSE VALIDATION")
print("-" * 50)
print("Current weather fields: PASSED")
print("Forecast availability: PASSED")
print("Hourly data availability: PASSED")
print("API response validation successful.")

API RESPONSE VALIDATION
--------------------------------------------------
Current weather fields: PASSED
Forecast availability: PASSED
Hourly data availability: PASSED
API response validation successful.


## 6. Data Processing and CSV Export

### 6.1 Objective

In this section, I convert the JSON response retrieved from WeatherAPI into a structured dataset using Pandas.

My goal is to organize hourly weather forecasts into a consistent tabular format, validate the data, and export the results as a CSV file.

### 6.2 Data Processing Method

I extract hourly forecast records from the API response. Each row represents one hourly forecast for Sioux Falls, South Dakota.

The dataset contains the following attributes:

| Column | Description |
|---|---|
| `location` | City name |
| `region` | State or region |
| `country` | Country name |
| `forecast_time` | Local forecast date and time |
| `temperature_c` | Forecast temperature in Celsius |
| `humidity_percent` | Relative humidity percentage |
| `wind_kph` | Wind speed in kilometers per hour |
| `precipitation_mm` | Forecast precipitation in millimeters |
| `condition` | Text description of weather conditions |

### 6.3 Data Validation

Before exporting the dataset, I check for missing values, invalid timestamps, duplicate records, and invalid numeric measurements.

These checks help me identify basic structural problems in the collected data.

### 6.4 Reproducibility

I save the dataset using a UTC timestamp in the filename. This allows me to distinguish between datasets collected during different executions.

For peer reviewers, the same code should generate a dataset with the same column structure. However, the weather values may differ because WeatherAPI updates its forecasts.

### 6.5 Ethical Transparency

The dataset contains weather forecasts rather than individual personal information.

I do not include API credentials in the exported CSV. I also document the source and collection methodology so another researcher can understand how the dataset was produced.

**Reference:** https://www.weatherapi.com/docs/

In [21]:
# Convert WeatherAPI JSON to Pandas DataFrame

import pandas as pd

records = []

for day in weather_data["forecast"]["forecastday"]:
    for hour in day["hour"]:

        records.append({
            "location": weather_data["location"]["name"],
            "region": weather_data["location"]["region"],
            "country": weather_data["location"]["country"],
            "forecast_time": hour["time"],
            "temperature_c": hour["temp_c"],
            "humidity_percent": hour["humidity"],
            "wind_kph": hour["wind_kph"],
            "precipitation_mm": hour["precip_mm"],
            "condition": hour["condition"]["text"]
        })

df = pd.DataFrame(records)

print("DataFrame created successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

DataFrame created successfully.
Rows: 24
Columns: 9


,location,region,country,forecast_time,temperature_c,humidity_percent,wind_kph,precipitation_mm,condition
0,Sioux Falls,South Dakota,United States of America,2026-10-24 00:00,20.0,50,30.2,0.00,Clear
1,Sioux Falls,South Dakota,United States of America,2026-10-24 01:00,18.7,55,26.6,0.00,Clear
2,Sioux Falls,South Dakota,United States of America,2026-10-24 02:00,18.7,53,26.3,0.00,Clear
3,Sioux Falls,South Dakota,United States of America,2026-10-24 03:00,18.1,55,24.8,0.01,Patchy rain nearby
4,Sioux Falls,South Dakota,United States of America,2026-10-24 04:00,17.6,57,23.8,0.01,Patchy rain nearby


In [22]:
# Validate Collected Weather Dataset

required_columns = [
    "location",
    "region",
    "country",
    "forecast_time",
    "temperature_c",
    "humidity_percent",
    "wind_kph",
    "precipitation_mm",
    "condition"
]

assert not df.empty, "Dataset is empty."

assert list(df.columns) == required_columns, (
    "Dataset columns do not match the expected schema."
)

assert df.isna().sum().sum() == 0, (
    "Missing values detected."
)

timestamps = pd.to_datetime(
    df["forecast_time"],
    errors="coerce"
)

assert timestamps.notna().all(), (
    "Invalid forecast timestamps."
)

assert timestamps.is_unique, (
    "Duplicate forecast timestamps."
)

numeric_columns = [
    "temperature_c",
    "humidity_percent",
    "wind_kph",
    "precipitation_mm"
]

for column in numeric_columns:
    assert pd.api.types.is_numeric_dtype(df[column]), (
        f"{column} is not numeric."
    )

assert df["humidity_percent"].between(0, 100).all(), (
    "Humidity is outside the valid range."
)

assert (df["wind_kph"] >= 0).all(), (
    "Negative wind speed detected."
)

assert (df["precipitation_mm"] >= 0).all(), (
    "Negative precipitation detected."
)

print("DATASET VALIDATION")
print("-" * 50)
print("Required columns: PASSED")
print("Missing values: PASSED")
print("Timestamp validation: PASSED")
print("Numeric validation: PASSED")
print("Measurement ranges: PASSED")
print("Dataset validation successful.")

DATASET VALIDATION
--------------------------------------------------
Required columns: PASSED
Missing values: PASSED
Timestamp validation: PASSED
Numeric validation: PASSED
Measurement ranges: PASSED
Dataset validation successful.


In [23]:
# Export Weather Dataset to CSV

from datetime import datetime, timezone

DATA.mkdir(parents=True, exist_ok=True)

collection_timestamp = datetime.now(
    timezone.utc
).strftime("%Y-%m-%dT%H%M%SZ")

csv_filename = f"weather_hourly_{collection_timestamp}.csv"

csv_path = DATA / csv_filename

df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8"
)

print("CSV exported successfully.")
print("Filename:", csv_path.name)
print("Rows:", len(df))
print("File size:", csv_path.stat().st_size, "bytes")

CSV exported successfully.
Filename: weather_hourly_2026-10-24T210736Z.csv
Rows: 24
File size: 2456 bytes


In [24]:
# Verify Saved CSV

saved_df = pd.read_csv(csv_path)

pd.testing.assert_frame_equal(
    df,
    saved_df,
    check_dtype=False
)

print("CSV verification successful.")
print("Saved rows:", len(saved_df))
print("Saved columns:", len(saved_df.columns))

display(saved_df.head())

CSV verification successful.
Saved rows: 24
Saved columns: 9


,location,region,country,forecast_time,temperature_c,humidity_percent,wind_kph,precipitation_mm,condition
0,Sioux Falls,South Dakota,United States of America,2026-10-24 00:00,20.0,50,30.2,0.00,Clear
1,Sioux Falls,South Dakota,United States of America,2026-10-24 01:00,18.7,55,26.6,0.00,Clear
2,Sioux Falls,South Dakota,United States of America,2026-10-24 02:00,18.7,53,26.3,0.00,Clear
3,Sioux Falls,South Dakota,United States of America,2026-10-24 03:00,18.1,55,24.8,0.01,Patchy rain nearby
4,Sioux Falls,South Dakota,United States of America,2026-10-24 04:00,17.6,57,23.8,0.01,Patchy rain nearby


In [25]:
# Inspect Generated Dataset Files

print("DATA DIRECTORY")
print("-" * 50)

for file in sorted(DATA.iterdir()):
    if file.is_file():
        print(file.name, "|", file.stat().st_size, "bytes")

assert csv_path.exists(), "CSV file not found."

print("-" * 50)
print("Dataset file verification passed.")

DATA DIRECTORY
--------------------------------------------------
weather_hourly_2026-10-24T210736Z.csv | 2456 bytes
--------------------------------------------------
Dataset file verification passed.


## 7. Data Provenance and Integrity Verification

### 7.1 Objective

In this section, I document the origin of my collected weather dataset and verify that the exported CSV has not been modified.

I use SHA-256 hashing to generate a checksum and create a JSON metadata file containing the information needed to understand and reproduce my experiment.

### 7.2 Data Provenance

Data provenance refers to information describing where data originated, how it was collected, and how it was processed.

For my experiment, I record:

- Data provider and API endpoint
- Geographic location and request parameters
- UTC collection timestamp
- Dataset filename and number of records
- Column names and file size
- Python and library versions
- SHA-256 checksum

I record only non-sensitive API parameters. Authentication keys and tokens are excluded.

### 7.3 SHA-256 Integrity Verification

I use the SHA-256 cryptographic hash function to calculate a checksum of my exported CSV file.

If another researcher has a copy of my original CSV and metadata, they can calculate the checksum independently and compare it with the recorded value.

A matching checksum confirms that the files are byte-for-byte identical. It does not independently prove that the weather forecasts are accurate.

### 7.4 Peer Reproducibility

If you are reviewing my notebook, execute the hashing and metadata cells after generating your own CSV.

You should verify that your calculated SHA-256 checksum matches the value recorded in your generated metadata file.

Because weather forecasts change over time, your checksum will likely differ from mine. This is expected and does not necessarily indicate a reproducibility failure.

### 7.5 Security and Ethical Considerations

I do not store my WeatherAPI key or GitHub token in provenance metadata.

I document the data source, processing procedure, and known limitations to support transparent and responsible research.

### References

- [NIST FIPS 180-4 — Secure Hash Standard](https://doi.org/10.6028/NIST.FIPS.180-4)
- [Python hashlib Documentation](https://docs.python.org/3/library/hashlib.html)
- [WeatherAPI Documentation](https://www.weatherapi.com/docs/)

In [26]:
# Generate SHA-256 Checksum

import hashlib

def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


csv_sha256 = calculate_sha256(csv_path)

print("SHA-256 checksum generated successfully.")
print("-" * 50)
print("Dataset:", csv_path.name)
print("SHA-256:", csv_sha256)

SHA-256 checksum generated successfully.
--------------------------------------------------
Dataset: weather_hourly_2026-10-24T210736Z.csv
SHA-256: 6bb88b668f192cf2e6e8f4d661a8d2a1b3c8b17acdd63faecaa9390d301c1f49


In [27]:
# Generate SHA-256 Checksum

import hashlib

def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


csv_sha256 = calculate_sha256(csv_path)

print("SHA-256 checksum generated successfully.")
print("-" * 50)
print("Dataset:", csv_path.name)
print("SHA-256:", csv_sha256)

SHA-256 checksum generated successfully.
--------------------------------------------------
Dataset: weather_hourly_2026-10-24T210736Z.csv
SHA-256: 6bb88b668f192cf2e6e8f4d661a8d2a1b3c8b17acdd63faecaa9390d301c1f49


In [28]:
# Generate Provenance Metadata

import json
import platform
from importlib.metadata import version

metadata = {
    "project": "CSC786 Ethical and Reproducible Data Collection",

    "data_source": {
        "provider": "WeatherAPI",
        "endpoint": ENDPOINT,
        "method": "GET",
        "documentation": "https://www.weatherapi.com/docs/",
        "parameters": SAFE_PARAMS,
        "authentication": "Google Colab Secrets"
    },

    "collection": {
        "timestamp_utc": collection_timestamp,
        "location": weather_data["location"]["name"],
        "region": weather_data["location"]["region"],
        "country": weather_data["location"]["country"],
        "forecast_date": weather_data["forecast"]["forecastday"][0]["date"]
    },

    "dataset": {
        "filename": csv_path.name,
        "relative_path": csv_path.relative_to(ROOT).as_posix(),
        "rows": len(df),
        "columns": list(df.columns),
        "size_bytes": csv_path.stat().st_size,
        "sha256": csv_sha256
    },

    "environment": {
        "python": platform.python_version(),
        "pandas": version("pandas"),
        "requests": version("requests")
    },

    "processing": {
        "source_format": "JSON",
        "output_format": "CSV",
        "method": "Hourly forecast extraction using pandas",
        "validation": "Required columns, timestamps, missing values and numeric ranges"
    },

    "limitations": [
        "Weather forecasts change over time.",
        "Live API responses may differ between executions.",
        "SHA-256 verifies file identity, not forecast accuracy."
    ]
}

metadata_path = DATA / f"{csv_path.stem}_metadata.json"

metadata_path.write_text(
    json.dumps(metadata, indent=4),
    encoding="utf-8"
)

print("Provenance metadata created successfully.")
print("Filename:", metadata_path.name)
print("Dataset:", csv_path.name)

Provenance metadata created successfully.
Filename: weather_hourly_2026-10-24T210736Z_metadata.json
Dataset: weather_hourly_2026-10-24T210736Z.csv


In [29]:
# Verify CSV and Metadata Integrity

import pandas as pd

saved_metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

saved_df = pd.read_csv(csv_path)

assert calculate_sha256(csv_path) == (
    saved_metadata["dataset"]["sha256"]
), "SHA-256 checksum mismatch."

assert len(saved_df) == (
    saved_metadata["dataset"]["rows"]
), "Row count mismatch."

assert list(saved_df.columns) == (
    saved_metadata["dataset"]["columns"]
), "Column mismatch."

assert csv_path.stat().st_size == (
    saved_metadata["dataset"]["size_bytes"]
), "File size mismatch."

print("DATA INTEGRITY VERIFICATION")
print("-" * 50)
print("SHA-256 checksum: PASSED")
print("Dataset row count: PASSED")
print("Column schema: PASSED")
print("File size: PASSED")
print("Dataset integrity verification successful.")

DATA INTEGRITY VERIFICATION
--------------------------------------------------
SHA-256 checksum: PASSED
Dataset row count: PASSED
Column schema: PASSED
File size: PASSED
Dataset integrity verification successful.


In [30]:
# Verify Metadata Security

metadata_text = metadata_path.read_text(
    encoding="utf-8"
)

assert WEATHERAPI_KEY not in metadata_text, (
    "WeatherAPI key detected in metadata."
)

assert "key" not in saved_metadata["data_source"]["parameters"], (
    "API authentication parameter found in metadata."
)

print("METADATA SECURITY CHECK")
print("-" * 50)
print("WeatherAPI key exposure: NOT DETECTED")
print("Authentication parameter: EXCLUDED")
print("Metadata security verification passed.")

METADATA SECURITY CHECK
--------------------------------------------------
WeatherAPI key exposure: NOT DETECTED
Authentication parameter: EXCLUDED
Metadata security verification passed.


In [31]:
# Inspect Generated Research Artifacts

print("RESEARCH DATA DIRECTORY")
print("-" * 60)

for file in sorted(DATA.iterdir()):
    if file.is_file():
        print(
            file.name,
            "|",
            file.stat().st_size,
            "bytes"
        )

print("-" * 60)

assert csv_path.exists(), "CSV file missing."
assert metadata_path.exists(), "Metadata file missing."

print("Research artifact verification passed.")

RESEARCH DATA DIRECTORY
------------------------------------------------------------
weather_hourly_2026-10-24T210736Z.csv | 2456 bytes
weather_hourly_2026-10-24T210736Z_metadata.json | 1831 bytes
------------------------------------------------------------
Research artifact verification passed.


## 8. Final Dataset Documentation and GitHub Version Control

### 8.1 Objective

In this section, I update my dataset documentation with the results of the data collection process.

I record the dataset's actual filename, collection timestamp, number of records, column structure, and SHA-256 checksum.

I also update my GitHub repository to preserve the documentation and reproducibility instructions.

### 8.2 Dataset Availability

I generate and verify the forecast CSV and metadata locally in Google Colab.

Because WeatherAPI imposes restrictions on caching forecast data, I do not permanently publish these generated files in my public repository.

Instead, I publish the notebook, methodology, environment requirements, and instructions needed for another student to generate their own dataset.

### 8.3 Instructions for Peer Reviewers

To reproduce my experiment:

1. Open my notebook in Google Colab.
2. Configure your own `WEATHERAPI_KEY` through Colab Secrets.
3. Execute the data collection and processing cells.
4. Generate your own CSV and provenance metadata.
5. Calculate the SHA-256 checksum of your CSV.
6. Verify that the calculated checksum matches your generated metadata.

The newly collected weather values and checksums may differ from mine because the API provides updated forecasts.

**Reference:** https://www.weatherapi.com/terms

In [32]:
# Update Dataset Documentation

data_readme = f"""# Dataset Documentation

## 1. Overview

I collected hourly weather forecast data for
Sioux Falls, South Dakota, using WeatherAPI.

Data Source: WeatherAPI
Endpoint: {ENDPOINT}
HTTP Method: GET

## 2. Collection Information

Collection Timestamp (UTC): {collection_timestamp}
Forecast Date: {weather_data["forecast"]["forecastday"][0]["date"]}

Location: {weather_data["location"]["name"]}
Region: {weather_data["location"]["region"]}
Country: {weather_data["location"]["country"]}

## 3. Dataset Summary

Generated CSV: {csv_path.name}

Total Records: {len(df)}
Total Columns: {len(df.columns)}

Missing Values: {int(df.isna().sum().sum())}

## 4. Dataset Schema

| Column | Description |
|---|---|
| location | City |
| region | State or region |
| country | Country |
| forecast_time | Local forecast timestamp |
| temperature_c | Temperature in Celsius |
| humidity_percent | Relative humidity percentage |
| wind_kph | Wind speed in kilometers per hour |
| precipitation_mm | Forecast precipitation in millimeters |
| condition | Weather condition |

## 5. Data Integrity

I generated a SHA-256 checksum for my local CSV.

SHA-256: {csv_sha256}

The checksum verifies the identity of the local
file used in this execution.

Peer reviewers must calculate and verify their
own checksum after generating their own CSV.

## 6. Provenance

I recorded the following in the local metadata JSON:

- WeatherAPI endpoint
- Non-sensitive request parameters
- UTC collection timestamp
- Dataset filename
- Dataset dimensions
- Python and package versions
- SHA-256 checksum

## 7. Reproduction Instructions

1. Open the notebook in Google Colab.
2. Configure WEATHERAPI_KEY in Colab Secrets.
3. Execute the environment setup.
4. Retrieve weather data.
5. Convert the JSON response into a DataFrame.
6. Validate and export the CSV.
7. Generate provenance metadata.
8. Verify the SHA-256 checksum.

GitHub publishing is not required to verify the
data collection workflow.

## 8. Limitations

Forecast values may change across executions.

The SHA-256 checksum does not establish the
accuracy of the weather forecasts.

Generated forecast data is kept locally,
subject to WeatherAPI's applicable terms.

## 9. References

WeatherAPI:
https://www.weatherapi.com/docs/

WeatherAPI Terms:
https://www.weatherapi.com/terms

Python hashlib:
https://docs.python.org/3/library/hashlib.html
"""

data_readme_path = ROOT / "DATA_README.md"

data_readme_path.write_text(
    data_readme,
    encoding="utf-8"
)

print("DATA_README.md updated successfully.")
print("Records documented:", len(df))
print("SHA-256 documented:", csv_sha256)

DATA_README.md updated successfully.
Records documented: 24
SHA-256 documented: 6bb88b668f192cf2e6e8f4d661a8d2a1b3c8b17acdd63faecaa9390d301c1f49


In [33]:
# Update .gitignore

gitignore_path = ROOT / ".gitignore"

existing = gitignore_path.read_text(encoding="utf-8")

if "data/" not in existing.splitlines():
    existing += "\n# Local forecast data\n" + "data/\n"

gitignore_path.write_text(
    existing,
    encoding="utf-8"
)

print(".gitignore updated.")
print("Local data/ directory is now ignored.")

.gitignore updated.
Local data/ directory is now ignored.


In [34]:
# Verify Files Before GitHub Push

import subprocess

required_files = [
    "README.md",
    "ETHICS.md",
    "DATA_README.md",
    "requirements.txt",
    ".gitignore"
]

for filename in required_files:
    assert (ROOT / filename).exists(), (
        f"Missing: {filename}"
    )

# Verify the WeatherAPI credential is not in documents
for filename in required_files:
    content = (ROOT / filename).read_text(
        encoding="utf-8"
    )
    assert WEATHERAPI_KEY not in content, (
        f"Credential found in {filename}"
    )

# Ensure the local dataset is not tracked or staged
tracked = subprocess.run(
    ["git", "ls-files", "--cached", "--", "data/"],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True
)

assert not tracked.stdout.strip(), (
    "Data files are already tracked by Git. "
    "Remove them from the Git index before pushing."
)

print("DOCUMENTATION VERIFICATION")
print("-" * 50)
print("Required files: PASSED")
print("Documentation credential check: PASSED")
print("Local dataset not tracked: PASSED")
print("Ready for documentation commit.")

DOCUMENTATION VERIFICATION
--------------------------------------------------
Required files: PASSED
Documentation credential check: PASSED
Local dataset not tracked: PASSED
Ready for documentation commit.


In [35]:
%cd {ROOT}

!git add README.md ETHICS.md DATA_README.md requirements.txt .gitignore
!git diff --cached --stat
!git commit -m "Update dataset provenance and reproducibility documentation"
!git push

/content
 .gitignore     |   3 ++
 DATA_README.md | 100 ++++++++++++++++++++++++++++++++++++---------------------
 2 files changed, 66 insertions(+), 37 deletions(-)
[main 91ee861] Update dataset provenance and reproducibility documentation
 2 files changed, 66 insertions(+), 37 deletions(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 1.66 KiB | 1.66 MiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/darkamgel/CSV-Reproducibilty.git
   70e5daa..91ee861  main -> main


## 9. GitHub Notebook Publication and Peer Reproduction

### 9.1 Objective

In this section, I publish my completed Google Colab notebook to GitHub and document how another student can reproduce my experiment independently.

My repository includes the notebook, Python dependencies, research documentation, ethical considerations, and dataset processing procedures.

### 9.2 Notebook Publication

I save my completed Google Colab notebook as an `.ipynb` file and upload it to my public GitHub repository.

Before publication, I verify that the notebook does not contain hardcoded credentials or sensitive execution outputs.

### 9.3 Instructions for Peer Reviewers

To reproduce my work:

1. Open my public GitHub repository.
2. Download the `.ipynb` notebook or open it directly in Google Colab.
3. Start a fresh Python 3 runtime.
4. Obtain your own WeatherAPI key from https://www.weatherapi.com/.
5. Configure `WEATHERAPI_KEY` using Google Colab Secrets.
6. Execute the environment setup and data collection cells.
7. Verify that the CSV dataset is generated successfully.
8. Generate the provenance metadata.
9. Recalculate and verify the SHA-256 checksum.
10. Compare your results with my documented methodology and expected dataset schema.

**Important:** The GitHub authentication and push cells are intended for my repository development workflow. Peer reviewers should skip these cells unless they are using their own GitHub repository and credentials.

### 9.4 Expected Results

I expect peer reviewers to generate a one-day hourly weather dataset, normally containing 24 records and nine columns.

They should also generate a metadata JSON file and successfully complete the dataset integrity checks.

The actual weather measurements and checksums may differ from mine because forecasts change over time.

### 9.5 Reproducibility Limitations

The notebook depends on an external API and an active WeatherAPI key. Network failures, API limits, and changes to the external service may affect execution.

I distinguish between reproducing the data collection procedure and obtaining identical weather measurements.

# Note: The code below need not to be run.

In [36]:
# Verify GitHub Repository

%cd {ROOT}

!git status --short
!git remote -v

/content
origin	https://github.com/darkamgel/CSV-Reproducibilty.git (fetch)
origin	https://github.com/darkamgel/CSV-Reproducibilty.git (push)


In [38]:
# finally pushing our code
%cd {ROOT}

!git add CSC786_Ethical_Reproducible_Data_Collection.ipynb
!git commit -m "Add reproducible Colab notebook"
!git push

/content
fatal: pathspec 'CSC786_Ethical_Reproducible_Data_Collection.ipynb' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
